In [1]:
import pandas as pd
from rdflib import Graph, URIRef, Literal, Namespace, RDF
import re

# 1. Setup
g = Graph()
EX = Namespace("http://example.org/kb/")
g.bind("ex", EX)

# 2. Chargement du CSV
df = pd.read_csv('extracted_knowledge.csv') # Assure-toi du nom du fichier

def to_camel_case(text):
    # Nettoie le texte pour en faire une URI valide sans espaces
    s = re.sub(r'[^a-zA-Z0-9\s]', '', str(text))
    return "".join(word.capitalize() for word in s.split())

# 3. Transformation en Triplets
for _, row in df.iterrows():
    # Sujet : L'entité
    subject_uri = URIRef(EX + to_camel_case(row['entity']))
    
    # Triplet 1: Définition du type (rdf:type)
    type_obj = URIRef(EX + to_camel_case(row['type']))
    g.add((subject_uri, RDF.type, type_obj))
    
    # Triplet 2: Source URL (Littéral)
    source_pred = URIRef(EX + "hasSource")
    source_lit = Literal(row['source_url'])
    g.add((subject_uri, source_pred, source_lit))

# 4. Export
g.serialize(destination='initial_kb.nt', format='nt')

print(f"Fichier créé : initial_kb.nt")
print(f"Nombre de triplets générés : {len(g)}")

Fichier créé : initial_kb.nt
Nombre de triplets générés : 819


c:\Users\François-Louis\AppData\Local\Programs\Python\Python312\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [2]:
def check_kb_thresholds(graph):
    """
    Vérifie si la base de connaissances respecte les minima du Step 1 :
    - >= 100 triplets
    - >= 50 entités
    """
    # Calcul des statistiques
    num_triplets = len(graph)
    
    # Une entité est soit un sujet, soit un objet de type URIRef (pas un littéral)
    subjects = {s for s in graph.subjects() if isinstance(s, URIRef)}
    objects = {o for o in graph.objects() if isinstance(o, URIRef)}
    entities = subjects.union(objects)
    num_entities = len(entities)

    print(f"--- Rapport de conformité (Step 1) ---")
    print(f"Triplets : {num_triplets} / 100")
    print(f"Entités  : {num_entities} / 50")
    
    # Diagnostic
    if num_triplets >= 100 and num_entities >= 50:
        print("✅ Statut : CONFORME. Vous pouvez passer au Step 2.")
    else:
        print("❌ Statut : INSUFFISANT.")
        if num_triplets < 100:
            print(f"   -> Il manque {100 - num_triplets} triplets.")
        if num_entities < 50:
            print(f"   -> Il manque {50 - num_entities} entités.")
    
    return num_triplets >= 100 and num_entities >= 50

# Utilisation après la construction du graphe 'g'
is_valid = check_kb_thresholds(g)

--- Rapport de conformité (Step 1) ---
Triplets : 819 / 100
Entités  : 397 / 50
✅ Statut : CONFORME. Vous pouvez passer au Step 2.


# Step 2



In [7]:
import requests
import pandas as pd
import re
import time
from rdflib import Graph, URIRef, Namespace, Literal
from rdflib.namespace import OWL, RDF, RDFS

# 1. Setup
g = Graph()
g.parse("initial_kb.nt", format="nt")
EX = Namespace("http://example.org/kb/")
mapping_data = []

# Headers pour éviter les erreurs API (bloquage par Wikidata)
headers = {
    'User-Agent': 'GeminiLabProject/1.0 (votre_email@example.com)'
}

def clean_uri_for_search(uri):
    local_name = str(uri).replace(str(EX), "")
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', local_name)

def get_wikidata_uri(label):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "search": label
    }
    try:
        # Ajout des headers et d'un timeout
        response = requests.get(url, params=params, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('search'):
                match = data['search'][0]
                return match['concepturi'], 0.99
        return None, 0.0
    except Exception as e:
        return None, 0.0

# 2. Processus d'alignement avec limitation de vitesse
entities = [s for s in g.subjects() if isinstance(s, URIRef) and str(s).startswith(str(EX))]
print(f"--- Alignement de {len(entities)} entités ---")

for ent in entities[:100]: # Testez d'abord sur les 100 premières pour gagner du temps
    search_query = clean_uri_for_search(ent)
    ext_uri, confidence = get_wikidata_uri(search_query)
    
    if ext_uri:
        print(f"✅ {search_query} -> {ext_uri}")
        g.add((ent, OWL.sameAs, URIRef(ext_uri)))
        mapping_data.append([str(ent), ext_uri, confidence])
    else:
        # Step 2: Créer une nouvelle entité et la définir semantiquement 
        print(f"❌ {search_query} (Définie localement)")
        g.add((ent, RDF.type, EX.ResearchEntity)) # Exemple de définition sémantique [cite: 52]
        mapping_data.append([str(ent), "Not Found", 0.0])
    
    time.sleep(0.2) # Pause de 200ms pour respecter l'API

# 3. Sauvegarde
mapping_df = pd.DataFrame(mapping_data, columns=["Private Entity", "External URI", "Confidence"])
mapping_df.to_csv("mapping_table.csv", index=False)
g.serialize(destination='aligned_kb.nt', format='nt')

--- Alignement de 819 entités ---
❌ East Faade Of The Louvre (Définie localement)
❌ Le Parthnon (Définie localement)
✅ Espagne -> http://www.wikidata.org/entity/Q29
✅ Golden Palace -> http://www.wikidata.org/entity/Q5579696
❌ Villa Krylos (Définie localement)
❌ Templo Mayor And The Coyolxauhqui Stone (Définie localement)
✅ Capferrat -> http://www.wikidata.org/entity/Q2974301
❌ Le Parthnon (Définie localement)
✅ Egypt -> http://www.wikidata.org/entity/Q79
✅ Bangkok -> http://www.wikidata.org/entity/Q1861
✅ Banque De France -> http://www.wikidata.org/entity/Q116205795
✅ London -> http://www.wikidata.org/entity/Q84
❌ Sernpre (Définie localement)
❌ The Monadnock Building (Définie localement)
❌ Ocrejaune (Définie localement)
❌ Wall Minbar Iwan Squinch (Définie localement)
✅ Art Nouveau -> http://www.wikidata.org/entity/Q34636
✅ Angkor Wat -> http://www.wikidata.org/entity/Q43473
❌ Historique Organise (Définie localement)
✅ Art -> http://www.wikidata.org/entity/Q8227356
❌ Church Of Stegenevi

<Graph identifier=Ndb15be7f11874d6c8af24fbf2e84234e (<class 'rdflib.graph.Graph'>)>

In [8]:
# Sérialisation en format Turtle
votre_graph_lisible = g.serialize(format="turtle")
print(votre_graph_lisible)

@prefix ns1: <http://example.org/kb/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ns1:11921316 a ns1:Loc ;
    ns1:hasSource "https://smarthistory.org/curated-guide/global-history-of-architecture-syllabus/" .

ns1:2DensificationAnd a ns1:Loc ;
    ns1:hasSource "https://www.academia.edu/96833138/The_Role_of_the_Fondation_Le_Corbusier_in_the_Conservation_of_the_Le_Corbusiers_Architectural_Work" .

ns1:Aachen a ns1:Loc ;
    ns1:hasSource "https://smarthistory.org/curated-guide/global-history-of-architecture-syllabus/" .

ns1:AcademiaMedicine a ns1:Org ;
    ns1:hasSource "https://www.academia.edu/96833138/The_Role_of_the_Fondation_Le_Corbusier_in_the_Conservation_of_the_Le_Corbusiers_Architectural_Work" .

ns1:AcadmieDesBeauxarts a ns1:Org ;
    ns1:hasSource "https://www.institutdefrance.fr/lepatrimoine/villa-grecque-kerylos/",
        "https://www.villa-ephrussi.com/fr/un-peu-dhistoire",
        "https://www.villakerylos.fr/la-villa-kerylos/" .

ns1:AcadmieDesInscriptionsEtBell

# Step 3 

In [9]:
from rdflib import Graph, URIRef, Namespace, Literal
from rdflib.namespace import OWL, RDF, RDFS, SKOS

# 1. Setup
g_onto = Graph()
EX = Namespace("http://example.org/kb/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
g_onto.bind("ex", EX)
g_onto.bind("wdt", WDT)
g_onto.bind("owl", OWL)

# 2. Définition de l'alignement (Exemples à adapter selon vos données)
# On aligne vos prédicats locaux sur des propriétés Wikidata standard
alignments = [
    (EX.hasSource, OWL.equivalentProperty, WDT.P1343), # P1343 = "described by source"
    (EX.Type, OWL.equivalentProperty, WDT.P31),       # P31 = "instance of"
    (EX.Location, OWL.equivalentProperty, WDT.P276),   # P276 = "location"
]

for s, p, o in alignments:
    g_onto.add((s, p, o))

# 3. Sauvegarde de l'ontologie (Livrable 2 du PDF)
g_onto.serialize(destination='ontology.ttl', format='turtle')

print("✅ Fichier ontology.ttl créé avec les alignements de prédicats.")

✅ Fichier ontology.ttl créé avec les alignements de prédicats.


# Step 4

In [10]:
import requests
import time
from rdflib import Graph, URIRef, Namespace, Literal, OWL

# 1. Setup
g_expanded = Graph()
# On charge votre KB alignée du Step 2
g_expanded.parse("aligned_kb.nt", format="nt")

EX = Namespace("http://example.org/kb/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
WD = Namespace("http://www.wikidata.org/entity/")

def query_wikidata_expansion(wd_uri):
    """
    Récupère le style architectural (P149) et la date de fondation (P571)
    pour une entité Wikidata donnée.
    """
    # On extrait l'ID (ex: Q1144650) de l'URI complète
    wd_id = wd_uri.split('/')[-1]
    
    query = f"""
    SELECT ?style ?styleLabel ?date WHERE {{
      OPTIONAL {{ wd:{wd_id} wdt:P149 ?style . }}
      OPTIONAL {{ wd:{wd_id} wdt:P571 ?date . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    url = "https://query.wikidata.org/sparql"
    try:
        r = requests.get(url, params={'query': query, 'format': 'json'}, timeout=10)
        return r.json()['results']['bindings']
    except:
        return []

# 2. Expansion 1-Hop
# On cherche toutes les entités qui ont un lien owl:sameAs vers Wikidata
entities_to_expand = list(g_expanded.triples((None, OWL.sameAs, None)))

print(f"Lancement de l'expansion sur {len(entities_to_expand)} entités...")

for local_ent, _, wd_uri in entities_to_expand[:50]: # Test sur les 50 premières
    results = query_wikidata_expansion(str(wd_uri))
    
    for res in results:
        # Ajout du Style
        if 'style' in res:
            style_uri = URIRef(res['style']['value'])
            g_expanded.add((local_ent, WDT.P149, style_uri))
            # Optionnel : ajouter le label du style pour l'humain
            if 'styleLabel' in res:
                g_expanded.add((style_uri, Namespace("http://www.w3.org/2000/01/rdf-schema#").label, Literal(res['styleLabel']['value'])))
        
        # Ajout de la Date
        if 'date' in res:
            date_lit = Literal(res['date']['value'])
            g_expanded.add((local_ent, WDT.P571, date_lit))
            
    time.sleep(0.5) # Pour ne pas être bloqué par le serveur SPARQL

# 3. Sauvegarde finale
g_expanded.serialize(destination='expanded_kb.nt', format='nt')
print(f"Expansion terminée ! Nouveau total de triplets : {len(g_expanded)}")

Lancement de l'expansion sur 59 entités...
Expansion terminée ! Nouveau total de triplets : 912


#### expansion to have min 50k triplets

In [13]:
import requests
import time
from rdflib import Graph, URIRef, Literal, Namespace

def robust_architecture_expansion(g_expanded, target_triples=60000):
    url = "https://query.wikidata.org/sparql"
    
    # On s'identifie auprès de Wikidata (Obligatoire pour éviter l'erreur char 0)
    headers = {
        'User-Agent': 'TD3_Architecture_Bot/1.0 (votre_email@example.com)',
        'Accept': 'application/sparql-results+json'
    }
    
    # Styles majeurs
    major_styles = ["Q186393", "Q37853", "Q843894", "Q54111", "Q12950", "Q174453"]
    
    WDT = Namespace("http://www.wikidata.org/prop/direct/")
    RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")

    for style_id in major_styles:
        if len(g_expanded) >= target_triples:
            print("Cible de 60k atteinte !")
            break
            
        print(f"Extraction pour le style WD:{style_id}...")
        
        # On demande par blocs de 5000 pour éviter le Timeout
        query = f"""
        SELECT ?building ?label ?country WHERE {{
          ?building wdt:P149 wd:{style_id} .
          ?building rdfs:label ?label .
          OPTIONAL {{ ?building wdt:P17 ?country . }}
          FILTER (lang(?label) = "en")
        }}
        LIMIT 10000
        """
        
        try:
            response = requests.get(url, params={'query': query, 'format': 'json'}, headers=headers, timeout=60)
            
            if response.status_code == 200:
                data = response.json()
                results = data['results']['bindings']
                
                for res in results:
                    b_uri = URIRef(res['building']['value'])
                    s_uri = URIRef(f"http://www.wikidata.org/entity/{style_id}")
                    
                    g_expanded.add((b_uri, WDT.P149, s_uri))
                    g_expanded.add((b_uri, RDFS.label, Literal(res['label']['value'])))
                    if 'country' in res:
                        g_expanded.add((b_uri, WDT.P17, URIRef(res['country']['value'])))
                
                print(f"  ✅ Succès : {len(results)} entités ajoutées. Total : {len(g_expanded)}")
            else:
                print(f"  ❌ Erreur HTTP {response.status_code}")
                
        except Exception as e:
            print(f"  ❌ Erreur technique : {e}")
        
        # Pause obligatoire pour ne pas être banni
        time.sleep(2)

    return g_expanded

# Exécution
g_expanded = robust_architecture_expansion(g_expanded)
print(f"\n--- SCORE FINAL : {len(g_expanded)} triplets ---")

Extraction pour le style WD:Q186393...
  ✅ Succès : 0 entités ajoutées. Total : 912
Extraction pour le style WD:Q37853...
  ✅ Succès : 1960 entités ajoutées. Total : 6765
Extraction pour le style WD:Q843894...
  ✅ Succès : 0 entités ajoutées. Total : 6765
Extraction pour le style WD:Q54111...
  ✅ Succès : 8961 entités ajoutées. Total : 33530
Extraction pour le style WD:Q12950...
  ✅ Succès : 0 entités ajoutées. Total : 33530
Extraction pour le style WD:Q174453...
  ✅ Succès : 0 entités ajoutées. Total : 33530

--- SCORE FINAL : 33530 triplets ---


# Step 4 ?


In [15]:
# Calcul des métriques demandées
num_triplets = len(g_expanded)
num_subjects = len(set(g_expanded.subjects()))
num_objects = len(set(o for o in g_expanded.objects() if isinstance(o, URIRef)))
num_entities = len(set(g_expanded.subjects()) | set(o for o in g_expanded.objects() if isinstance(o, URIRef)))
num_properties = len(set(g_expanded.predicates()))

print("--- STATISTIQUES POUR LE RAPPORT ---")
print(f"1. Nombre total de triplets : {num_triplets}")
print(f"2. Nombre d'entités uniques : {num_entities}")
print(f"3. Nombre de propriétés (relations) : {num_properties}")
print(f"4. Nombre de sujets : {num_subjects}")
print(f"5. Nombre d'objets (URIs) : {num_objects}")

--- STATISTIQUES POUR LE RAPPORT ---
1. Nombre total de triplets : 33530
2. Nombre d'entités uniques : 11435
3. Nombre de propriétés (relations) : 6
4. Nombre de sujets : 11251
5. Nombre d'objets (URIs) : 184


In [16]:
# Top 5 des propriétés les plus utilisées
from collections import Counter
preds = [str(p) for p in g_expanded.predicates()]
top_preds = Counter(preds).most_common(5)

print("--- ANALYSE DE LA STRUCTURE ---")
for p, count in top_preds:
    print(f"Propriété: {p} | Utilisée {count} fois")

--- ANALYSE DE LA STRUCTURE ---
Propriété: http://www.wikidata.org/prop/direct/P17 | Utilisée 10887 fois
Propriété: http://www.wikidata.org/prop/direct/P149 | Utilisée 10875 fois
Propriété: http://www.w3.org/2000/01/rdf-schema#label | Utilisée 10856 fois
Propriété: http://www.w3.org/1999/02/22-rdf-syntax-ns#type | Utilisée 433 fois
Propriété: http://example.org/kb/hasSource | Utilisée 420 fois


In [17]:
g_expanded.serialize(destination='final_kb.nt', format='nt')
print("Fichier final_kb.nt généré avec succès.")

Fichier final_kb.nt généré avec succès.
